In [1]:
!pip install umap-learn pandas scikit-learn matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import os

def load_data():
    train_in = pd.read_csv('train_in.csv', header=None).values
    train_out = pd.read_csv('train_out.csv', header=None).values.flatten()
    test_in = pd.read_csv('test_in.csv', header=None).values
    test_out = pd.read_csv('test_out.csv', header=None).values.flatten()
    return train_in, train_out, test_in, test_out

def part1_centers_and_distances(train_in, train_out):
    print("T1P1")
    centers = []
    for d in range(10):
        Cd = train_in[train_out == d]
        cd = np.mean(Cd, axis=0)
        centers.append(cd)
    centers = np.array(centers)
    
    dist_matrix = cdist(centers, centers, metric='euclidean')
    print("Distance Matrix between centers:")
    print(np.round(dist_matrix, 2))
    
    #  getting top 3 closest pairs
    dists = []
    for i in range(10):
        for j in range(i+1, 10):
            dists.append((dist_matrix[i, j], i, j))
    dists.sort()
    print("Top 3 closest pairs (most difficult to separate):")
    for k in range(3):
        print(f"digit {dists[k][1]} and digit {dists[k][2]} (distance: {dists[k][0]:.2f})")
        
    return centers

def part2_dimensionality_reduction(train_in, train_out):
    print("T1P2")
    # PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(train_in)
    
    # TSNE
    tsne = TSNE(n_components=2, random_state=42)
    tsne_result = tsne.fit_transform(train_in)
    
    # UMAP
    # suppressing numba warnings if any
    import warnings
    from numba.core.errors import NumbaDeprecationWarning
    warnings.simplefilter('ignore', category=NumbaDeprecationWarning)
    
    umap_model = umap.UMAP(n_components=2, random_state=42)
    umap_result = umap_model.fit_transform(train_in)
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    scatter = axes[0].scatter(pca_result[:, 0], pca_result[:, 1], c=train_out, cmap='tab10', s=5)
    axes[0].set_title('PCA')
    
    axes[1].scatter(tsne_result[:, 0], tsne_result[:, 1], c=train_out, cmap='tab10', s=5)
    axes[1].set_title('t-SNE')
    
    axes[2].scatter(umap_result[:, 0], umap_result[:, 1], c=train_out, cmap='tab10', s=5)
    axes[2].set_title('U-MAP')
    
    # Add a single colorbar
    fig.colorbar(scatter, ax=axes.ravel().tolist(), label='Digit')
    plt.savefig('task1_part2_dim_reduction.png')
    print("Saved dimensionality reduction plots to task1_part2_dim_reduction.png")
    plt.close()

def part3_nearest_mean_classifier(train_in, train_out, test_in, test_out, centers):
    print("T1P3")
    
    def predict(X, centers):
        dists = cdist(X, centers, metric='euclidean')
        return np.argmin(dists, axis=1)
        
    train_preds = predict(train_in, centers)
    test_preds = predict(test_in, centers)
    
    train_acc = accuracy_score(train_out, train_preds)
    test_acc = accuracy_score(test_out, test_preds)
    
    print(f"Nearest Mean Classifier - Train Accuracy: {train_acc*100:.2f}%")
    print(f"Nearest Mean Classifier - Test Accuracy: {test_acc*100:.2f}%")
    
    return train_preds, test_preds

def part4_knn_classifier(train_in, train_out, test_in, test_out, nearest_mean_test_preds):
    print("T1P4")
    # Using k=1 as a naive KNN 
    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(train_in, train_out)
    
    train_preds = knn.predict(train_in)
    test_preds = knn.predict(test_in)
    
    train_acc = accuracy_score(train_out, train_preds)
    test_acc = accuracy_score(test_out, test_preds)
    
    print(f"KNN (k=1) Classifier - Train Accuracy: {train_acc*100:.2f}%")
    print(f"KNN (k=1) Classifier - Test Accuracy: {test_acc*100:.2f}%")
    
    # Confusion Matrices
    cm_nm = confusion_matrix(test_out, nearest_mean_test_preds)
    cm_knn = confusion_matrix(test_out, test_preds)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    sns.heatmap(cm_nm, annot=True, fmt='d', ax=axes[0], cmap='Blues')
    axes[0].set_title('Nearest Mean Confusion Matrix')
    axes[0].set_xlabel('Predicted Label')
    axes[0].set_ylabel('True Label')
    
    sns.heatmap(cm_knn, annot=True, fmt='d', ax=axes[1], cmap='Blues')
    axes[1].set_title('KNN (k=1) Confusion Matrix')
    axes[1].set_xlabel('Predicted Label')
    axes[1].set_ylabel('True Label')
    
    plt.savefig('task1_part4_confusion_matrices.png')
    print("Saved confusion matrices to task1_part4_confusion_matrices.png")
    plt.close()

if __name__ == "__main__":
    train_in, train_out, test_in, test_out = load_data()
    centers = part1_centers_and_distances(train_in, train_out)
    part2_dimensionality_reduction(train_in, train_out)
    _, nm_test_preds = part3_nearest_mean_classifier(train_in, train_out, test_in, test_out, centers)
    part4_knn_classifier(train_in, train_out, test_in, test_out, nm_test_preds)


T1P1
Distance Matrix between centers:
[[ 0.   14.45  9.33  9.14 10.77  7.52  8.15 11.86  9.91 11.49]
 [14.45  0.   10.13 11.73 10.17 11.12 10.61 10.74 10.09  9.93]
 [ 9.33 10.13  0.    8.18  7.93  7.91  7.33  8.87  7.08  8.89]
 [ 9.14 11.73  8.18  0.    9.09  6.12  9.3   8.92  7.02  8.35]
 [10.77 10.17  7.93  9.09  0.    8.    8.78  7.58  7.38  6.01]
 [ 7.52 11.12  7.91  6.12  8.    0.    6.7   9.21  6.97  8.26]
 [ 8.15 10.61  7.33  9.3   8.78  6.7   0.   10.89  8.59 10.44]
 [11.86 10.74  8.87  8.92  7.58  9.21 10.89  0.    8.47  5.43]
 [ 9.91 10.09  7.08  7.02  7.38  6.97  8.59  8.47  0.    6.4 ]
 [11.49  9.93  8.89  8.35  6.01  8.26 10.44  5.43  6.4   0.  ]]
Top 3 closest pairs (most difficult to separate):
digit 7 and digit 9 (distance: 5.43)
digit 4 and digit 9 (distance: 6.01)
digit 3 and digit 5 (distance: 6.12)
T1P2


/home/s5026652/.local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved dimensionality reduction plots to task1_part2_dim_reduction.png
T1P3
Nearest Mean Classifier - Train Accuracy: 86.35%
Nearest Mean Classifier - Test Accuracy: 80.40%
T1P4
KNN (k=1) Classifier - Train Accuracy: 100.00%
KNN (k=1) Classifier - Test Accuracy: 91.50%
Saved confusion matrices to task1_part4_confusion_matrices.png
